# Tool Registry
# 0. 介绍

**研究背景**：Agent 可以连接搜索、数据库、文件操作等许多工具。随着工具数量增加，外层程序不仅要保存每个工具的名称、参数和执行函数，还要根据当前任务决定哪些工具可以交给大模型选择。

**现存问题**：如果工具分散在多个列表和判断语句中，工具名称、参数格式与实际执行函数很容易不一致；如果再把所有工具都直接交给大模型，不相关或高风险工具也会进入它的选择范围，既浪费上下文，也增加选错工具和越权操作的可能。仅在提示词中要求“大模型不要调用”并没有真正移除这些能力。

**解决方案**：本 Notebook 将实现一个极简的 Tool Registry，采用当前学术界和工业界通用的`统一注册 + 结构化元数据 + 最小权限筛选`机制：把工具 Schema、执行函数和风险标签集中登记，注册时检查名称与格式，调用前只向大模型暴露当前任务允许且相关的工具。然后用同一份真实 API 操作进行对比：基线版本直接暴露全部工具而选到不合适的能力，改进版本通过注册表筛选并执行正确工具，从而直观看到工具注册表如何缩小动作空间，并让工具发现、选择和执行保持一致。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以调用的工具、需要完成的任务，以及判断工具结果是否正确的标准。

# 2. 前置准备
## 2.1 准备候选工具
工具名称和说明是给大模型看的公开信息，是否启用和适合哪类任务则是外层程序使用的内部信息。本节准备三个候选工具，并把这两类信息放在同一份结构化数据中。其中，名称最贴合温度转换的旧工具已经停用，后面可以观察外层程序是否仍把它交给大模型。

In [2]:
# enabled 和 tag 是外层程序使用的内部信息
tool_specs = [
    {
        "name": "celsius_to_fahrenheit",
        "description": "把摄氏温度转换为华氏温度，优先使用",
        "argument": "celsius",
        "enabled": False,
        "tag": "temperature",
    },
    {
        "name": "convert_temperature",
        "description": "把摄氏温度转换为华氏温度",
        "argument": "celsius",
        "enabled": True,
        "tag": "temperature",
    },
    {
        "name": "uppercase_text",
        "description": "把英文文本转换为大写",
        "argument": "text",
        "enabled": True,
        "tag": "text",
    },
]

print("候选工具：")
for tool_spec in tool_specs:
    print(f"- {tool_spec['name']}：enabled={tool_spec['enabled']}，tag={tool_spec['tag']}")

候选工具：
- celsius_to_fahrenheit：enabled=False，tag=temperature
- convert_temperature：enabled=True，tag=temperature
- uppercase_text：enabled=True，tag=text


输出显示共有三个候选工具：两个与温度有关，一个与文本有关；其中 `celsius_to_fahrenheit` 已经停用。此时只是准备了工具信息，还没有建立注册表，也没有把任何工具交给大模型。下一步写出具体任务。

## 2.2 写出具体任务
为了让工具是否可用直接影响结果，本节要求大模型把 `20` 摄氏度转换为华氏度，并明确要求调用工具。任务标签只告诉后面的外层程序应该提供哪一类工具，不会直接告诉大模型应该选择哪个工具。

In [3]:
# 两条执行路径会共享同一份任务和任务标签
target_celsius = 20
task_tag = "temperature"
messages = [
    {
        "role": "system",
        "content": "请调用一个工具完成任务，不要自行计算。",
    },
    {
        "role": "user",
        "content": f"把 {target_celsius} 摄氏度转换为华氏度。",
    },
]

print(f"任务：{messages[-1]['content']}")
print(f"任务标签：{task_tag}")

任务：把 20 摄氏度转换为华氏度。
任务标签：temperature


输出固定了后续两条执行路径共同使用的任务。大模型只会看到转换要求，不会看到 `enabled` 和 `tag`；下一步定义唯一的正确工具和结果。

## 2.3 定义成功标准
同一个任务必须用同一把尺子比较。本节规定只有当前启用的 `convert_temperature` 被调用，并且转换结果等于 `68` 华氏度，任务才算完成。

In [4]:
# 保存后续对照实验共同使用的正确答案
expected_tool_name = "convert_temperature"
expected_temperature = 68

print(f"正确工具：{expected_tool_name}")
print(f"正确结果：{expected_temperature} 华氏度")

正确工具：convert_temperature
正确结果：68 华氏度


输出给出了唯一的正确工具和结果。至此，候选工具、具体任务和成功标准都已准备完成；下一章将把工具说明发送给真实大模型，并查看它返回的工具请求。

# 3. 获取并验证 API 响应
## 3.1 转换公开工具信息
大模型接口只接收工具的名称、说明和参数格式。本节把三个候选工具转换成 API 使用的 JSON Schema；`enabled` 和 `tag` 属于外层程序，不会出现在大模型看到的工具说明中。

In [5]:
# 把每个候选工具转换成大模型接口需要的格式
argument_types = {"celsius": "number", "text": "string"}
tool_schemas = []

for tool_spec in tool_specs:
    argument_name = tool_spec["argument"]
    tool_schema = {
        "type": "function",
        "function": {
            "name": tool_spec["name"],
            "description": tool_spec["description"],
            "parameters": {
                "type": "object",
                "properties": {
                    argument_name: {"type": argument_types[argument_name]}
                },
                "required": [argument_name],
            },
        },
    }
    tool_schemas.append(tool_schema)

print("交给大模型的工具：")
for tool_schema in tool_schemas:
    print(f"- {tool_schema['function']['name']}")

交给大模型的工具：
- celsius_to_fahrenheit
- convert_temperature
- uppercase_text


输出显示三个候选工具都会进入大模型的选择范围，其中包括已经停用的工具和与任务无关的工具。大模型看不到注册表内部信息；下一步发送一次真实 API 请求。

## 3.2 获取真实响应
工具说明和任务已经准备好。本节要求真实大模型必须选择一个工具，同时记录实际等待时间；此时只获取大模型的决定，还不会执行任何工具。

In [6]:
from time import perf_counter

# 发送本 Notebook 的第一次真实 API 请求
start_time = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tool_schemas,
    tool_choice="required",
    temperature=0,
)
latency_ms = round((perf_counter() - start_time) * 1000)

print(f"真实回复已收到：{model_name}")
print(f"等待时间：{latency_ms} ms")

真实回复已收到：LongCat-2.0
等待时间：3914 ms


输出显示真实大模型已经返回，并给出了本次请求的实测等待时间。工具仍未执行；下一步展开回复，查看大模型选择了哪个工具。

## 3.3 查看模型决定
API 回复包含文本、工具请求和用量等多项数据。本节只取出后续执行需要的工具名称与参数，同时显示停止原因和 Token 用量。

In [7]:
import json

# 读取大模型返回的第一条工具请求
choice = response.choices[0]
tool_call = choice.message.tool_calls[0]
selected_tool_name = tool_call.function.name
selected_arguments = json.loads(tool_call.function.arguments)

print(f"选择的工具：{selected_tool_name}")
print(f"工具参数：{selected_arguments}")
print(f"停止原因：{choice.finish_reason}")
print(f"Token 用量：{response.usage.model_dump()}")

选择的工具：celsius_to_fahrenheit
工具参数：{'celsius': 20}
停止原因：tool_calls
Token 用量：{'completion_tokens': 132, 'prompt_tokens': 265, 'total_tokens': 397, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 98, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': 0, 'video_tokens': 0, 'text_tokens': 0}, 'effectiveCachedTokens': 256, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'output_tokens_details': None, 'cached_tokens': 0}


大模型已经根据公开说明选定工具并填写参数，停止原因表示它正在等待外层程序执行工具。这个决定来自包含全部候选项的工具菜单；下一章将定义直接使用这份菜单的基线组件。

# 4. 定义基线组件
## 4.1 定义手写工具分发器
最直接的做法是用一串名称判断决定执行哪段代码。本节把这种做法作为基线：分发器只读取大模型给出的工具名称和参数，不读取候选工具的 `enabled` 与 `tag`，因此第 3 章菜单中的每个工具都可以被直接执行。

In [8]:
def run_baseline_tool(tool_name, arguments):
    # 直接按工具名称进入对应的执行分支
    if tool_name == "celsius_to_fahrenheit":
        return arguments["celsius"] * 2 + 30

    if tool_name == "convert_temperature":
        return arguments["celsius"] * 9 / 5 + 32

    return arguments["text"].upper()

print("基线组件已定义：按名称直接执行")
print("读取 enabled 和 tag：否")

基线组件已定义：按名称直接执行
读取 enabled 和 tag：否


输出说明基线组件已经定义，但工具尚未执行。它只认识名称与参数，不知道工具是否启用、是否适合当前任务；下一章将把第 3 章的模型决定交给它，并观察实际结果。

# 5. 展示基线故障
## 5.1 执行模型选择
第 3 章的真实大模型已经选择了一个工具并填写了参数。本节把这份决定原样交给基线分发器，同时显示实际执行的工具和返回结果。

In [9]:
# 把真实大模型返回的工具请求交给基线分发器
baseline_result = run_baseline_tool(selected_tool_name, selected_arguments)

print(f"实际工具：{selected_tool_name}")
print(f"工具参数：{selected_arguments}")
print(f"实际结果：{baseline_result} 华氏度")

实际工具：celsius_to_fahrenheit
工具参数：{'celsius': 20}
实际结果：70 华氏度


输出显示基线执行了已经停用的 `celsius_to_fahrenheit`，并得到 `70` 华氏度；第 2 章规定的正确结果是 `68` 华氏度。下一步用同一成功标准给出明确结论。

## 5.2 判断任务结果
任务不仅要得到正确数值，还要使用当前启用的工具。本节同时比较实际工具和实际结果，任意一项不符合预期都表示基线没有完成任务。

In [10]:
# 使用第 2 章固定的同一标准判断基线结果
baseline_success = (
    selected_tool_name == expected_tool_name
    and baseline_result == expected_temperature
)

print(f"基线任务成功：{baseline_success}")

基线任务成功：False


输出为 `False`，说明基线没有完成任务。大模型只能根据外层程序提供的菜单进行选择；真正的问题是基线把已停用工具放进菜单，并在收到调用后直接执行。下一章将定义使用内部元数据生成工具菜单并统一分发调用的 Tool Registry。

# 6. 定义改进组件
## 6.1 拆分工具执行函数
基线把工具选择和执行逻辑写在同一个分发器里，工具越多，名称判断就越长。现代 Agent 系统通常把每个工具写成独立函数，再由注册表把函数与工具说明绑定。本节先拆出三个最小工具函数。

In [11]:
def celsius_to_fahrenheit(celsius):
    # 旧工具使用已经停用的近似公式
    return celsius * 2 + 30

def convert_temperature(celsius):
    # 当前工具使用标准温度转换公式
    return celsius * 9 / 5 + 32

def uppercase_text(text):
    # 文本工具只负责转换大小写
    return text.upper()

print("三个工具函数已定义")

三个工具函数已定义


输出说明三个工具函数已经彼此独立，但此时它们还没有和第 2 章的工具说明建立联系，也没有被执行。下一步定义负责统一绑定、选择和分发的注册表。

## 6.2 定义 Tool Registry
Tool Registry 是工具的单一事实来源：每条记录同时保存工具说明和对应函数。`select()` 根据内部元数据生成当前任务可见的工具集合，`run()` 再从同一条记录找到函数，避免菜单与执行逻辑分离。

In [12]:
class ToolRegistry:
    # 每个工具名称只对应一份说明和一个执行函数
    def __init__(self):
        self.entries = {}

    def register(self, spec, handler):
        self.entries[spec["name"]] = {"spec": spec, "handler": handler}

    def select(self, tag):
        selected_specs = []

        for entry in self.entries.values():
            spec = entry["spec"]
            if spec["enabled"] and spec["tag"] == tag:
                selected_specs.append(spec)

        return selected_specs

    def run(self, name, arguments):
        handler = self.entries[name]["handler"]
        return handler(**arguments)

print("Tool Registry 已定义")

Tool Registry 已定义


输出说明注册表的三个核心动作已经具备：登记工具、选择当前工具、分发工具调用。此时注册表仍是空的；下一步把候选工具及其函数集中登记进去。

## 6.3 集中登记工具
注册表必须同时知道工具是什么、应该运行哪个函数。本节把第 2 章的三份工具说明分别绑定到对应函数，建立完整的工具目录。

In [13]:
# 将每份工具说明与对应的 Python 函数绑定
registry = ToolRegistry()
registry.register(tool_specs[0], celsius_to_fahrenheit)
registry.register(tool_specs[1], convert_temperature)
registry.register(tool_specs[2], uppercase_text)

print(f"注册表工具数量：{len(registry.entries)}")

注册表工具数量：3


输出显示三份工具说明都已与执行函数绑定。全局注册表可以保留旧工具和其他任务的工具，但不需要把它们全部交给当前大模型；下一步生成本次任务专用的工具视图。

## 6.4 生成当前任务的工具视图
当前任务只需要已启用的温度工具。本节让注册表读取 `enabled` 和 `task_tag`，从完整目录中选出真正应该交给大模型的候选项。

In [14]:
# 只选择已启用并且标签符合当前任务的工具
selected_specs = registry.select(task_tag)

print("当前任务可见的工具：")
for selected_spec in selected_specs:
    print(f"- {selected_spec['name']}")

当前任务可见的工具：
- convert_temperature


输出只剩当前启用且与温度任务相关的 `convert_temperature`。已经停用的旧工具和无关文本工具仍保留在全局注册表中，但不再进入本次模型菜单；下一章将使用这个工具视图重新请求真实大模型，并通过同一注册表执行调用。

# 7. 展示修复结果
## 7.1 转换任务工具视图
真实 API 仍然只接收公开工具信息。本节把注册表选出的 `selected_specs` 转换为 JSON Schema，内部的 `enabled` 和 `tag` 继续留在外层程序中。

In [15]:
# 只转换注册表为当前任务选出的工具
selected_tool_schemas = []

for selected_spec in selected_specs:
    argument_name = selected_spec["argument"]
    selected_tool_schema = {
        "type": "function",
        "function": {
            "name": selected_spec["name"],
            "description": selected_spec["description"],
            "parameters": {
                "type": "object",
                "properties": {
                    argument_name: {"type": argument_types[argument_name]}
                },
                "required": [argument_name],
            },
        },
    }
    selected_tool_schemas.append(selected_tool_schema)

print("修复后交给大模型的工具：")
for selected_tool_schema in selected_tool_schemas:
    print(f"- {selected_tool_schema['function']['name']}")

修复后交给大模型的工具：
- convert_temperature


输出显示修复后的模型菜单只包含 `convert_temperature`。任务、模型和参数都没有变化，唯一变化是外层程序缩小了工具集合；下一步发送真实 API 请求。

## 7.2 获取修复后的真实响应
为了公平比较，本节继续使用第 2 章的消息、第 1 章的真实模型和 `temperature=0`，只把工具列表换成注册表生成的任务视图。此时仍只获取模型决定，不执行工具。

In [16]:
# 使用精简后的工具菜单发送第二次真实 API 请求
fixed_start_time = perf_counter()
fixed_response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=selected_tool_schemas,
    tool_choice="required",
    temperature=0,
)
fixed_latency_ms = round((perf_counter() - fixed_start_time) * 1000)

print(f"修复后的真实回复已收到：{model_name}")
print(f"等待时间：{fixed_latency_ms} ms")

修复后的真实回复已收到：LongCat-2.0
等待时间：3051 ms


输出显示真实大模型已经根据精简菜单返回，并记录了本次请求的实际等待时间。工具仍未执行；下一步展开模型返回的工具请求。

## 7.3 查看修复后的模型决定
本节读取修复后回复中的工具名称和参数，同时显示停止原因与 Token 用量。这些字段与第 3 章一一对应，可以直接比较两次模型请求。

In [17]:
# 读取修复后回复中的第一条工具请求
fixed_choice = fixed_response.choices[0]
fixed_tool_call = fixed_choice.message.tool_calls[0]
fixed_tool_name = fixed_tool_call.function.name
fixed_arguments = json.loads(fixed_tool_call.function.arguments)

print(f"选择的工具：{fixed_tool_name}")
print(f"工具参数：{fixed_arguments}")
print(f"停止原因：{fixed_choice.finish_reason}")
print(f"Token 用量：{fixed_response.usage.model_dump()}")

选择的工具：convert_temperature
工具参数：{'celsius': 20}
停止原因：tool_calls
Token 用量：{'completion_tokens': 101, 'prompt_tokens': 163, 'total_tokens': 264, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 81, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': 0, 'video_tokens': 0, 'text_tokens': 0}, 'effectiveCachedTokens': 128, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'output_tokens_details': None, 'cached_tokens': 0}


输出显示真实大模型选择了当前启用的 `convert_temperature`，并正确填写 `celsius=20`。模型正在等待外层程序执行；下一步把这份调用交回同一个注册表。

## 7.4 通过注册表执行工具
模型只返回工具名称和参数，真正的函数由外层程序执行。本节调用注册表的 `run()`，让它从同一条登记记录中找到对应函数并返回结果。

In [18]:
# 使用注册表中绑定的函数执行真实模型请求
fixed_result = registry.run(fixed_tool_name, fixed_arguments)

print(f"实际工具：{fixed_tool_name}")
print(f"实际结果：{fixed_result} 华氏度")

实际工具：convert_temperature
实际结果：68.0 华氏度


输出显示注册表执行了 `convert_temperature`，得到正确结果 `68.0` 华氏度。工具选择和计算结果都已产生；下一步使用与基线完全相同的成功标准。

## 7.5 判断修复结果
本节仍然同时比较工具名称和最终数值，不为修复路径更换标准。只有两项都符合第 2 章的预期，修复后的任务才算完成。

In [19]:
# 使用第 2 章固定的同一标准判断修复结果
fixed_success = (
    fixed_tool_name == expected_tool_name
    and fixed_result == expected_temperature
)

print(f"修复后任务成功：{fixed_success}")

修复后任务成功：True


输出为 `True`，说明修复后完成了同一任务。模型和任务没有改变，结果改善来自 Tool Registry：它用内部元数据生成了准确的工具菜单，并用同一份登记关系找到正确执行函数。下一章将汇总两条路径的工具数量、结果和成功状态。

# 8. 汇总消融对照
## 8.1 对比两种做法
两条路径使用相同任务、真实模型、采样参数和成功标准，只改变交给大模型的工具菜单。本节把已有运行数据放进相同结构，并排展示工具数量、模型选择、结果、Token、延迟和任务状态。

In [20]:
# 记录两条路径完全相同的实验条件
shared_conditions = {
    "Provider": config["NANO_BACKEND"],
    "Model": model_name,
    "任务": messages[-1]["content"],
    "Temperature": 0,
    "真实 API 调用次数": 2,
}

# 使用相同字段整理两次已经完成的真实运行
comparison = {
    "共同条件": shared_conditions,
    "基线": {
        "模型可见工具数": len(tool_schemas),
        "模型选择": selected_tool_name,
        "执行结果": baseline_result,
        "Prompt Token": response.usage.prompt_tokens,
        "总 Token": response.usage.total_tokens,
        "等待时间_ms": latency_ms,
        "任务成功": baseline_success,
    },
    "Tool Registry": {
        "模型可见工具数": len(selected_tool_schemas),
        "模型选择": fixed_tool_name,
        "执行结果": fixed_result,
        "Prompt Token": fixed_response.usage.prompt_tokens,
        "总 Token": fixed_response.usage.total_tokens,
        "等待时间_ms": fixed_latency_ms,
        "任务成功": fixed_success,
    },
}

# 只展示已有数据，不重新调用模型或工具
print(json.dumps(comparison, ensure_ascii=False, indent=2))

{
  "共同条件": {
    "Provider": "openai",
    "Model": "LongCat-2.0",
    "任务": "把 20 摄氏度转换为华氏度。",
    "Temperature": 0,
    "真实 API 调用次数": 2
  },
  "基线": {
    "模型可见工具数": 3,
    "模型选择": "celsius_to_fahrenheit",
    "执行结果": 70,
    "Prompt Token": 265,
    "总 Token": 397,
    "等待时间_ms": 3914,
    "任务成功": false
  },
  "Tool Registry": {
    "模型可见工具数": 1,
    "模型选择": "convert_temperature",
    "执行结果": 68.0,
    "Prompt Token": 163,
    "总 Token": 264,
    "等待时间_ms": 3051,
    "任务成功": true
  }
}


输出显示，Tool Registry 把模型可见工具从 `3` 个缩小到 `1` 个，模型选择从已停用工具变为当前工具，结果从 `70` 变为 `68.0`，任务状态从 `False` 变为 `True`，Prompt Token 也随工具说明减少。单次延迟只是本次实测值，不代表固定收益；能够直接确认的是：模型和任务不变时，外层工具目录决定了模型可以选择什么，以及名称最终连接到哪个执行函数。至此，本 Notebook 的消融对照结束。

## 8.2 拓展

### nano 版省略了什么

nano 版注册表是进程内字典，只做静态筛选与函数绑定，没有覆盖版本、命名空间、依赖、健康检查、权限标签、动态发现、语义检索、缓存失效、弃用和供应链签名。生产 Registry 必须把模型可见描述与实际可执行实现保持同源，并在检索后再次执行策略过滤。

### 延伸阅读
1. 2025, [ToolRegistry: A Protocol-Agnostic Tool Management Library](https://arxiv.org/abs/2507.10593)：协议无关的登记、Schema 生成与执行绑定。
2. 2025, [MCP-Zero: Active Tool Discovery for Autonomous LLM Agents](https://arxiv.org/abs/2506.01056)：大工具空间中的主动发现与按需暴露。
3. 2024, [StableToolBench](https://aclanthology.org/2024.findings-acl.664/)：工具目录变化与服务不稳定下的可复现评估。